# 第 10 天 · Function calling 原理

> 📓 **这是第 10 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/10)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai pydantic python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 10 天 · Function calling 原理：查天气（假数据）的完整往返
没有 Key 也能跑（假模型照真实协议返回 tool_calls）；填了 LLM_API_KEY 就调用真模型。
往返流程：用户提问 → 模型返回 tool_calls → 我们执行函数 → 以 tool 角色发回结果 → 模型给最终回答"""
import asyncio
import json
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv()


# ---------- 1. 真正的函数（类比 Java 里的 Service 方法），用假数据 ----------
FAKE_WEATHER = {
    "北京": {"condition": "晴", "temp_c": 12, "humidity": 30},
    "上海": {"condition": "小雨", "temp_c": 18, "humidity": 85},
    "深圳": {"condition": "多云", "temp_c": 26, "humidity": 70},
}


class WeatherArgs(BaseModel):
    """参数用 pydantic 定义：既能生成 JSON Schema 给模型看，又能校验模型给的参数（复用第 9 天）"""
    city: str = Field(description="城市中文名，如 北京、上海")
    unit: str = Field(default="celsius", pattern="^(celsius|fahrenheit)$", description="温度单位")


def get_weather(city: str, unit: str = "celsius") -> dict:
    data = FAKE_WEATHER.get(city)
    if data is None:
        return {"error": f"没有 {city} 的天气数据，支持：{', '.join(FAKE_WEATHER)}"}
    temp = data["temp_c"] if unit == "celsius" else round(data["temp_c"] * 9 / 5 + 32, 1)
    return {"city": city, "condition": data["condition"], "temp": temp,
            "unit": unit, "humidity": data["humidity"]}


# ---------- 2. tools 参数：用 JSON Schema 把函数"说明书"交给模型（类比 OpenAPI 文档） ----------
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询某个城市当前的天气。用户问天气、穿衣、要不要带伞时调用。",
        "parameters": WeatherArgs.model_json_schema(),
    },
}]

# 函数名 → (参数模型, 实现)：类比 Spring 里按 bean 名字找 Bean
REGISTRY = {"get_weather": (WeatherArgs, get_weather)}


def run_tool(name: str, arguments: str) -> str:
    """执行一次工具调用。返回值一定是字符串（tool 消息的 content 只能是文本）"""
    if name not in REGISTRY:
        return json.dumps({"error": f"未知工具 {name}"}, ensure_ascii=False)
    args_model, func = REGISTRY[name]
    try:
        args = args_model.model_validate_json(arguments)  # arguments 是 JSON 字符串，不可信
    except ValidationError as e:
        return json.dumps({"error": f"参数不合法：{e.errors()[0]['msg']}"}, ensure_ascii=False)
    return json.dumps(func(**args.model_dump()), ensure_ascii=False)


# ---------- 3. 模型：统一返回 dict 形式的 assistant 消息 ----------
async def fake_llm(messages: list[dict], tools: list[dict]) -> dict:
    """按真实协议模拟：还没拿到工具结果时返回 tool_calls；拿到了就组织最终回答"""
    await asyncio.sleep(0.05)
    tool_results = [m for m in messages if m["role"] == "tool"]
    if not tool_results:
        return {"role": "assistant", "content": None, "tool_calls": [{
            "id": "call_001", "type": "function",
            "function": {"name": "get_weather",
                         "arguments": json.dumps({"city": "上海"}, ensure_ascii=False)},
        }]}
    w = json.loads(tool_results[-1]["content"])
    tip = "记得带伞" if "雨" in w["condition"] else "不用带伞"
    return {"role": "assistant",
            "content": f"上海现在{w['condition']}，{w['temp']}°C，湿度 {w['humidity']}%，出门{tip}。"}


async def real_llm(messages: list[dict], tools: list[dict]) -> dict:
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    resp = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"),
        messages=messages, tools=tools, temperature=0,
    )
    msg = resp.choices[0].message
    out = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:  # SDK 对象 → 普通 dict，方便打印和放回 messages
        out["tool_calls"] = [{"id": c.id, "type": "function",
                              "function": {"name": c.function.name, "arguments": c.function.arguments}}
                             for c in msg.tool_calls]
    return out


# ---------- 4. 一次完整往返 ----------
async def ask(question: str, llm) -> str:
    messages = [
        {"role": "system", "content": "你是出行助手。涉及天气必须调用工具查询，不要编造。"},
        {"role": "user", "content": question},
    ]
    print(f"① 用户：{question}")

    reply = await llm(messages, TOOLS)                      # 第 1 次调用：带上 tools
    if not reply.get("tool_calls"):                         # 模型觉得不需要工具，直接回答
        return reply["content"]
    messages.append(reply)                                  # ⚠️ 带 tool_calls 的 assistant 消息必须放回去
    for call in reply["tool_calls"]:
        name, arguments = call["function"]["name"], call["function"]["arguments"]
        print(f"② 模型要求调用：{name}({arguments})   id={call['id']}")
        result = run_tool(name, arguments)                  # 真正执行函数的是"我们"，不是模型
        print(f"③ 执行结果：{result}")
        messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})

    final = await llm(messages, TOOLS)                      # 第 2 次调用：模型看到结果后作答
    print("④ 最终回答：", end="")
    return final["content"]


async def main():
    llm = real_llm if os.getenv("LLM_API_KEY") else fake_llm
    print("使用", "真模型" if llm is real_llm else "假模型（没填 Key）", "\n")
    print(await ask("我等下要去上海出差，要带伞吗？", llm))

    print("\n--- 直接测一下 run_tool 的容错 ---")
    print(run_tool("get_weather", '{"city": "杭州"}'))
    print(run_tool("get_weather", '{"city": "北京", "unit": "kelvin"}'))
    print(run_tool("get_stock", "{}"))


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await